# Rural Water-Point Operational Status Prediction

## Stage 4 – Data Preprocessing and Feature Engineering

### Objective

The objective of this stage is to prepare the dataset for machine-learning modelling based on the findings from the previous exploratory data analysis.

The preprocessing focuses on:

- handling missing and invalid values,
- identifying and handling duplicate or redundant information,
- treating outliers where justified,
- encoding categorical variables appropriately,
- scaling or transforming numerical variables where required,
- performing meaningful feature engineering,
- performing feature selection where appropriate,
- and preventing information leakage during preprocessing.

All important preprocessing decisions will be justified using evidence from the dataset and the findings from Stage 3.

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

## 1. Load and Verify the Raw Dataset

The predictor variables and target labels are provided in separate files.

The two files are merged using the unique `id` variable to create the labelled dataset required for supervised learning.

The raw data is kept unchanged so that all preprocessing decisions remain reproducible.

In [3]:
values = pd.read_csv("../data/raw/training_set_values.csv")
labels = pd.read_csv("../data/raw/training_set_labels.csv")

print("Values shape:", values.shape)
print("Labels shape:", labels.shape)

Values shape: (59400, 40)
Labels shape: (59400, 2)


### 1.1 Merge Predictor and Target Data

The predictor variables and target labels are stored in separate files. They are merged using the unique `id` variable.

A one-to-one merge is enforced to ensure that each water-point observation is matched with exactly one target label and that the merge does not unintentionally duplicate observations.

In [4]:
df = values.merge(
    labels,
    on="id",
    how="inner",
    validate="one_to_one"
)

print("Merged dataset shape:", df.shape)

Merged dataset shape: (59400, 41)


### 1.2 Verify the Dataset Merge

The merged dataset is checked to confirm that:

- the number of observations is correct,
- each `id` is unique,
- no target values are missing,
- and no duplicate IDs were introduced during the merge.

In [5]:
print("Number of rows:", len(df))
print("Unique IDs:", df["id"].nunique())
print("Missing target values:", df["status_group"].isna().sum())
print("Duplicate IDs:", df["id"].duplicated().sum())

Number of rows: 59400
Unique IDs: 59400
Missing target values: 0
Duplicate IDs: 0


## 2. Define Predictors and Target

The target variable `status_group` is separated from the predictor variables before further preprocessing.

`status_group` represents the operational status of each water point and is the variable that the machine-learning models will predict.

All remaining variables are initially retained as candidate predictors. Feature removal and selection will be performed later based on the findings from Stage 3 EDA.

In [6]:
X = df.drop(columns=["status_group"])
y = df["status_group"]

print("Predictor shape:", X.shape)
print("Target shape:", y.shape)

Predictor shape: (59400, 40)
Target shape: (59400,)


In [7]:
print("Target classes:")
print(y.unique())

print("\nTarget distribution:")
print(y.value_counts())

print("\nTarget distribution (%):")
print(
    y.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Target classes:
<StringArray>
['functional', 'non functional', 'functional needs repair']
Length: 3, dtype: str

Target distribution:
status_group
functional                 32259
non functional             22824
functional needs repair     4317
Name: count, dtype: int64

Target distribution (%):
status_group
functional                 54.31
non functional             38.42
functional needs repair     7.27
Name: proportion, dtype: float64


## 3. Train-Test Split

The labelled dataset is divided into training and testing subsets before data-dependent preprocessing is performed.

A stratified split is used because the target variable contains an imbalanced class distribution. Stratification helps preserve approximately the same proportion of each target class in both subsets.

The training set will be used to learn preprocessing parameters and develop the models. The test set will remain unseen until final model evaluation.

This separation helps prevent information leakage.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (47520, 40)
Testing shape: (11880, 40)


### 3.1 Verify Target Distribution After Splitting

The target-class distribution is compared between the training and testing sets to verify that stratification has preserved the original class proportions.

Similar class proportions are expected in both subsets.

In [9]:
print("Original target distribution (%):")
print(
    y.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nTraining target distribution (%):")
print(
    y_train.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nTesting target distribution (%):")
print(
    y_test.value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Original target distribution (%):
status_group
functional                 54.31
non functional             38.42
functional needs repair     7.27
Name: proportion, dtype: float64

Training target distribution (%):
status_group
functional                 54.31
non functional             38.42
functional needs repair     7.27
Name: proportion, dtype: float64

Testing target distribution (%):
status_group
functional                 54.31
non functional             38.43
functional needs repair     7.26
Name: proportion, dtype: float64


In [10]:
print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

print("\nTraining target distribution (%):")
print(y_train.value_counts(normalize=True).mul(100).round(2))

print("\nTesting target distribution (%):")
print(y_test.value_counts(normalize=True).mul(100).round(2))

Training shape: (47520, 40)
Testing shape: (11880, 40)

Training target distribution (%):
status_group
functional                 54.31
non functional             38.42
functional needs repair     7.27
Name: proportion, dtype: float64

Testing target distribution (%):
status_group
functional                 54.31
non functional             38.43
functional needs repair     7.26
Name: proportion, dtype: float64


## 4. Duplicate and Invalid Record Handling

The exploratory analysis identified potential duplicate records and unusual values that may affect model development.

Before removing any observations, the training data is examined to determine:

- whether complete duplicate observations exist,
- whether duplicate identifiers exist,
- whether records contain impossible or invalid values,
- and whether suspicious values represent missing information or legitimate observations.

Only records that can be justified as invalid or redundant will be removed or transformed.

In [11]:
duplicate_train = X_train.duplicated().sum()

print("Exact duplicate predictor rows in training data:", duplicate_train)

Exact duplicate predictor rows in training data: 0


In [12]:
duplicate_test = X_test.duplicated().sum()

print("Exact duplicate predictor rows in testing data:", duplicate_test)

Exact duplicate predictor rows in testing data: 0


In [13]:
print("Duplicate IDs in training data:", X_train["id"].duplicated().sum())
print("Duplicate IDs in testing data:", X_test["id"].duplicated().sum())

Duplicate IDs in training data: 0
Duplicate IDs in testing data: 0


### 4.1 Investigation of Zero-Valued Numerical Features

Several numerical variables contain zero values.

A zero value may represent a genuine observation for some variables, while for other variables it may represent unavailable or unrecorded information.

Therefore, zero-valued observations are quantified before deciding whether they require transformation.

In [14]:
numeric_columns = X_train.select_dtypes(
    include=np.number
).columns

zero_summary = pd.DataFrame({
    "zero_count": [
        (X_train[col] == 0).sum()
        for col in numeric_columns
    ],
    "zero_percentage": [
        (X_train[col] == 0).mean() * 100
        for col in numeric_columns
    ]
}, index=numeric_columns)

zero_summary.sort_values(
    "zero_percentage",
    ascending=False
).round(2)

,zero_count,zero_percentage
num_private,46932,98.76
amount_tsh,33265,70.00
population,17066,35.91
construction_year,16517,34.76
gps_height,16305,34.31
longitude,1442,3.03
district_code,19,0.04
id,1,0.00
latitude,0,0.00
region_code,0,0.00


### 4.2 Invalid Construction-Year Values

The `construction_year` variable contains zero values.

A construction year of zero is not a valid calendar year and is therefore treated as an unavailable construction year rather than a genuine numerical observation.

The variable will be retained because valid construction years contain useful information for feature engineering.

In [15]:
print(
    "Training records with construction_year = 0:",
    (X_train["construction_year"] == 0).sum()
)

print(
    "Percentage:",
    round(
        (X_train["construction_year"] == 0).mean() * 100,
        2
    ),
    "%"
)

Training records with construction_year = 0: 16517
Percentage: 34.76 %


### 4.3 Invalid Geographic Coordinates

The EDA identified observations where longitude is equal to zero and latitude is extremely close to zero.

These coordinates are treated as likely unavailable geographic information rather than valid locations.

The records will not be deleted because other predictor information may still be useful. Instead, the invalid coordinate values will be converted to missing values and handled through the later preprocessing pipeline.

In [16]:
invalid_geo_train = (
    (X_train["longitude"] == 0) &
    (X_train["latitude"].abs() < 0.001)
)

invalid_geo_test = (
    (X_test["longitude"] == 0) &
    (X_test["latitude"].abs() < 0.001)
)

print(
    "Invalid geographic records - training:",
    invalid_geo_train.sum()
)

print(
    "Invalid geographic records - testing:",
    invalid_geo_test.sum()
)

Invalid geographic records - training: 1442
Invalid geographic records - testing: 370


In [17]:
print("Exact duplicate predictor rows - training:", X_train.duplicated().sum())
print("Exact duplicate predictor rows - testing:", X_test.duplicated().sum())

print("Duplicate IDs - training:", X_train["id"].duplicated().sum())
print("Duplicate IDs - testing:", X_test["id"].duplicated().sum())

Exact duplicate predictor rows - training: 0
Exact duplicate predictor rows - testing: 0
Duplicate IDs - training: 0
Duplicate IDs - testing: 0


### 4.4 Investigation of Near-Constant Features

The EDA identified `num_private` as a highly zero-dominated numerical feature, with approximately 98.77% of training observations equal to zero.

Rather than removing the feature solely because of its distribution, its number of unique values and relationship with the target are examined before making a final feature-selection decision.

In [18]:
print("Unique values in num_private:")
print(X_train["num_private"].value_counts().head(20))

print("\nNumber of unique values:")
print(X_train["num_private"].nunique())

print("\nNon-zero percentage:")
print(
    round(
        (X_train["num_private"] != 0).mean() * 100,
        2
    ),
    "%"
)

Unique values in num_private:
num_private
0      46932
6         60
1         59
8         40
5         37
32        35
39        24
45        23
7         22
2         21
15        20
93        19
3         18
47        17
4         16
102       16
65        16
20        13
25        11
80        11
Name: count, dtype: int64

Number of unique values:
59

Non-zero percentage:
1.24 %


In [19]:
print("construction_year = 0:")
print((X_train["construction_year"] == 0).sum())

print(
    "Percentage:",
    round(
        (X_train["construction_year"] == 0).mean() * 100,
        2
    ),
    "%"
)

print("\nValid construction-year range:")
print(
    X_train.loc[
        X_train["construction_year"] > 0,
        "construction_year"
    ].agg(["min", "max"])
)

construction_year = 0:
16517
Percentage: 34.76 %

Valid construction-year range:
min    1960
max    2013
Name: construction_year, dtype: int64


In [20]:
invalid_geo_train = (
    (X_train["longitude"] == 0) &
    (X_train["latitude"].abs() < 0.001)
)

invalid_geo_test = (
    (X_test["longitude"] == 0) &
    (X_test["latitude"].abs() < 0.001)
)

print(
    "Invalid geographic records - training:",
    invalid_geo_train.sum()
)

print(
    "Invalid geographic records - testing:",
    invalid_geo_test.sum()
)

Invalid geographic records - training: 1442
Invalid geographic records - testing: 370


## 5. Initial Feature Removal

The `id` variable is removed from the modelling predictors because it serves as an observation identifier rather than representing a meaningful characteristic of the water point.

Keeping an identifier could allow a model to learn arbitrary record-specific patterns rather than generalizable relationships.

The removal is applied consistently to both training and testing predictors.

In [21]:
X_train = X_train.drop(columns=["id"])
X_test = X_test.drop(columns=["id"])

print("Training shape after ID removal:", X_train.shape)
print("Testing shape after ID removal:", X_test.shape)

Training shape after ID removal: (47520, 39)
Testing shape after ID removal: (11880, 39)


### 5.1 Treatment of Invalid Construction Years

The EDA identified 16,517 training observations with `construction_year = 0`, representing 34.76% of the training data.

A value of zero is not a valid construction year because the observed valid construction years range from 1960 to 2013.

Therefore, zero is treated as unavailable information and converted to `NaN`.

The feature is retained because valid construction-year information can be used for feature engineering.

In [22]:
X_train["construction_year"] = (
    X_train["construction_year"]
    .replace(0, np.nan)
)

X_test["construction_year"] = (
    X_test["construction_year"]
    .replace(0, np.nan)
)

print(
    "Missing construction_year - training:",
    X_train["construction_year"].isna().sum()
)

print(
    "Missing construction_year - testing:",
    X_test["construction_year"].isna().sum()
)

Missing construction_year - training: 16517
Missing construction_year - testing: 4192


### 5.2 Treatment of Invalid Geographic Coordinates

The EDA identified records where `longitude = 0` and `latitude` is approximately zero.

These values are treated as unavailable geographic information rather than valid geographic coordinates.

The affected observations are retained because their remaining predictor information may still be useful. Only the invalid coordinate values are replaced with missing values.

The resulting missing values will be handled later through the preprocessing pipeline.

In [23]:
invalid_geo_train = (
    (X_train["longitude"] == 0) &
    (X_train["latitude"].abs() < 0.001)
)

invalid_geo_test = (
    (X_test["longitude"] == 0) &
    (X_test["latitude"].abs() < 0.001)
)

X_train.loc[
    invalid_geo_train,
    ["longitude", "latitude"]
] = np.nan

X_test.loc[
    invalid_geo_test,
    ["longitude", "latitude"]
] = np.nan

print(
    "Missing longitude after treatment - training:",
    X_train["longitude"].isna().sum()
)

print(
    "Missing latitude after treatment - training:",
    X_train["latitude"].isna().sum()
)

Missing longitude after treatment - training: 1442
Missing latitude after treatment - training: 1442


In [24]:
print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

print(
    "Missing construction_year - training:",
    X_train["construction_year"].isna().sum()
)

print(
    "Missing construction_year - testing:",
    X_test["construction_year"].isna().sum()
)

print(
    "Missing longitude - training:",
    X_train["longitude"].isna().sum()
)

print(
    "Missing latitude - training:",
    X_train["latitude"].isna().sum()
)

Training shape: (47520, 39)
Testing shape: (11880, 39)
Missing construction_year - training: 16517
Missing construction_year - testing: 4192
Missing longitude - training: 1442
Missing latitude - training: 1442


## 6. Missing-Value Analysis

Missing values are investigated systematically after the initial treatment of invalid values.

The training dataset is used to determine the missingness profile because preprocessing decisions must be based only on the training data.

Missing values are examined separately for numerical and categorical variables because the appropriate treatment may differ between these variable types.

No blanket row deletion is applied because removing observations with any missing value could unnecessarily reduce the available training data.

In [25]:
missing_summary = pd.DataFrame({
    "missing_count": X_train.isna().sum(),
    "missing_percentage": X_train.isna().mean().mul(100)
})

missing_summary = (
    missing_summary
    .query("missing_count > 0")
    .sort_values("missing_percentage", ascending=False)
)

missing_summary.round(2)

,missing_count,missing_percentage
scheme_name,23050,48.51
construction_year,16517,34.76
scheme_management,3129,6.58
installer,2917,6.14
funder,2906,6.12
public_meeting,2644,5.56
permit,2443,5.14
longitude,1442,3.03
latitude,1442,3.03
subvillage,286,0.60


### 6.1 Missing Values by Variable Type

Numerical and categorical variables are examined separately because they require different preprocessing strategies.

Numerical variables may require statistical imputation, while categorical variables can potentially use an explicit missing category such as `"Unknown"`.

The choice will be made based on the amount and meaning of missing information in each feature.

In [26]:
numeric_features = X_train.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X_train.select_dtypes(
    exclude=np.number
).columns.tolist()

print("Number of numerical features:", len(numeric_features))
print("Number of categorical features:", len(categorical_features))

print("\nNumerical features with missing values:")
print(
    missing_summary.loc[
        missing_summary.index.isin(numeric_features)
    ].round(2)
)

print("\nCategorical features with missing values:")
print(
    missing_summary.loc[
        missing_summary.index.isin(categorical_features)
    ].round(2)
)

Number of numerical features: 9
Number of categorical features: 30

Numerical features with missing values:
                   missing_count  missing_percentage
construction_year          16517               34.76
longitude                   1442                3.03
latitude                    1442                3.03

Categorical features with missing values:
                   missing_count  missing_percentage
scheme_name                23050               48.51
scheme_management           3129                6.58
installer                   2917                6.14
funder                      2906                6.12
public_meeting              2644                5.56
permit                      2443                5.14
subvillage                   286                0.60
wpt_name                       2                0.00


### 6.2 Missingness Across Observations

The number of missing predictor values per observation is examined to determine whether missing information is concentrated in a small number of records or distributed across the dataset.

In [27]:
row_missing = X_train.isna().sum(axis=1)

print("Minimum missing values in a row:", row_missing.min())
print("Maximum missing values in a row:", row_missing.max())
print("Average missing values per row:", round(row_missing.mean(), 2))

print("\nRows with 0 missing values:")
print((row_missing == 0).sum())

print("\nRows with more than 5 missing values:")
print((row_missing > 5).sum())

print("\nRows with more than 10 missing values:")
print((row_missing > 10).sum())

Minimum missing values in a row: 0
Maximum missing values in a row: 8
Average missing values per row: 1.19

Rows with 0 missing values:
17551

Rows with more than 5 missing values:
30

Rows with more than 10 missing values:
0


## 7. Categorical Feature Cardinality

The dataset contains a large number of categorical variables. Their cardinality is examined before selecting an encoding strategy.

Low-cardinality variables can generally be encoded using one-hot encoding, while high-cardinality variables may require rare-category grouping, alternative encoding, or removal where the feature provides limited generalizable information.

Cardinality is examined using the training data only to avoid using information from the held-out test set when making preprocessing decisions.

In [28]:
categorical_cardinality = pd.DataFrame({
    "unique_values": X_train[categorical_features].nunique(dropna=False),
    "missing_count": X_train[categorical_features].isna().sum(),
    "missing_percentage": (
        X_train[categorical_features].isna().mean() * 100
    )
})

categorical_cardinality = (
    categorical_cardinality
    .sort_values("unique_values", ascending=False)
)

categorical_cardinality.round(2)

,unique_values,missing_count,missing_percentage
wpt_name,30661,2,0.00
subvillage,17232,286,0.60
scheme_name,2563,23050,48.51
ward,2082,0,0.00
installer,1930,2917,6.14
funder,1716,2906,6.12
date_recorded,349,0,0.00
lga,124,0,0.00
region,21,0,0.00
extraction_type,18,0,0.00


### 7.1 High-Cardinality Feature Investigation

Several categorical variables contain a large number of unique categories.

High-cardinality variables are examined because direct one-hot encoding can create a very large number of predictors and may result in sparse representations with limited generalization.

The most important high-cardinality candidates identified during EDA are investigated before the final encoding strategy is selected.

In [29]:
high_cardinality_candidates = [
    "wpt_name",
    "subvillage",
    "ward",
    "installer",
    "funder",
    "scheme_name"
]

high_cardinality_summary = pd.DataFrame({
    "unique_values": X_train[high_cardinality_candidates].nunique(dropna=False),
    "missing_count": X_train[high_cardinality_candidates].isna().sum(),
    "missing_percentage": (
        X_train[high_cardinality_candidates].isna().mean() * 100
    )
}).sort_values("unique_values", ascending=False)

high_cardinality_summary.round(2)

,unique_values,missing_count,missing_percentage
wpt_name,30661,2,0.00
subvillage,17232,286,0.60
scheme_name,2563,23050,48.51
ward,2082,0,0.00
installer,1930,2917,6.14
funder,1716,2906,6.12


### 7.2 Removal of Constant Features

The `recorded_by` feature contains only one unique value across the training data.

Because a constant feature provides no variation between observations, it cannot help a supervised learning model distinguish between target classes.

Therefore, `recorded_by` is removed from the modelling predictors.

In [30]:
print("Unique values in recorded_by:")
print(X_train["recorded_by"].nunique())

print("\nValue counts:")
print(X_train["recorded_by"].value_counts())

Unique values in recorded_by:
1

Value counts:
recorded_by
GeoData Consultants Ltd    47520
Name: count, dtype: int64


In [31]:
X_train = X_train.drop(columns=["recorded_by"])
X_test = X_test.drop(columns=["recorded_by"])

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (47520, 38)
Testing shape: (11880, 38)


### 7.3 High-Cardinality Feature Investigation

High-cardinality categorical variables can produce very large encoded feature spaces when one-hot encoding is applied.

To assess whether these variables behave more like meaningful categorical predictors or identifier-like variables, their number of unique categories is compared with the number of training observations.

The proportion of observations represented by unique categories is also examined.

In [32]:
high_cardinality_summary = pd.DataFrame({
    "unique_values": X_train[high_cardinality_candidates].nunique(dropna=False),
    "unique_ratio_percent": (
        X_train[high_cardinality_candidates]
        .nunique(dropna=False)
        .div(len(X_train))
        .mul(100)
    ),
    "missing_count": X_train[high_cardinality_candidates].isna().sum(),
    "missing_percentage": (
        X_train[high_cardinality_candidates]
        .isna()
        .mean()
        .mul(100)
    )
}).sort_values("unique_values", ascending=False)

high_cardinality_summary.round(2)

,unique_values,unique_ratio_percent,missing_count,missing_percentage
wpt_name,30661,64.52,2,0.00
subvillage,17232,36.26,286,0.60
scheme_name,2563,5.39,23050,48.51
ward,2082,4.38,0,0.00
installer,1930,4.06,2917,6.14
funder,1716,3.61,2906,6.12


### 7.4 Rare-Category Investigation

The frequency of categorical levels is examined for high-cardinality variables.

A large number of categories occurring only once or a few times may indicate that direct one-hot encoding would create a sparse and poorly generalizable representation.

This analysis will inform whether rare categories should be grouped or whether some identifier-like variables should be excluded.

In [33]:
for col in high_cardinality_candidates:
    counts = X_train[col].value_counts(dropna=False)

    print(f"\n--- {col} ---")
    print("Unique categories:", counts.size)
    print("Categories occurring once:", (counts == 1).sum())
    print("Categories occurring <= 5 times:", (counts <= 5).sum())
    print("Largest category frequency:", counts.iloc[0])


--- wpt_name ---
Unique categories: 30661
Categories occurring once: 27224
Categories occurring <= 5 times: 30254
Largest category frequency: 2879

--- subvillage ---
Unique categories: 17232
Categories occurring once: 8948
Categories occurring <= 5 times: 15838
Largest category frequency: 420

--- ward ---
Unique categories: 2082
Categories occurring once: 53
Categories occurring <= 5 times: 316
Largest category frequency: 257

--- installer ---
Unique categories: 1930
Categories occurring once: 971
Categories occurring <= 5 times: 1436
Largest category frequency: 13978

--- funder ---
Unique categories: 1716
Categories occurring once: 868
Categories occurring <= 5 times: 1251
Largest category frequency: 7321

--- scheme_name ---
Unique categories: 2563
Categories occurring once: 709
Categories occurring <= 5 times: 1561
Largest category frequency: 23050


### 7.5 Treatment of the Date Variable

`date_recorded` contains date information rather than an ordinary nominal category.

Therefore, it will not be directly one-hot encoded as a categorical variable.

Instead, date components such as the recording year and month will be investigated as potential engineered features.

This preserves temporal information while avoiding the creation of hundreds of unrelated dummy variables.

In [34]:
print("Training date range:")
print(X_train["date_recorded"].min(), "to", X_train["date_recorded"].max())

print("\nTesting date range:")
print(X_test["date_recorded"].min(), "to", X_test["date_recorded"].max())

Training date range:
2002-10-14 to 2013-12-03

Testing date range:
2004-05-01 to 2013-12-03


In [35]:
X_train["date_recorded"] = pd.to_datetime(
    X_train["date_recorded"],
    errors="coerce"
)

X_test["date_recorded"] = pd.to_datetime(
    X_test["date_recorded"],
    errors="coerce"
)

print("Training date dtype:", X_train["date_recorded"].dtype)
print("Testing date dtype:", X_test["date_recorded"].dtype)

Training date dtype: datetime64[us]
Testing date dtype: datetime64[us]


### 7.5.1 Date-Based Feature Engineering

The recording date is transformed into separate temporal features.

The recording year captures the period in which the water-point observation was collected, while the recording month may capture seasonal or operational differences.

These engineered variables provide a more useful representation of the original date information for subsequent modelling.

In [36]:
X_train["recorded_year"] = X_train["date_recorded"].dt.year
X_train["recorded_month"] = X_train["date_recorded"].dt.month

X_test["recorded_year"] = X_test["date_recorded"].dt.year
X_test["recorded_month"] = X_test["date_recorded"].dt.month

print(
    X_train[
        ["date_recorded", "recorded_year", "recorded_month"]
    ].head()
)

      date_recorded  recorded_year  recorded_month
43360    2011-07-27           2011               7
7263     2011-03-23           2011               3
2486     2011-03-07           2011               3
313      2011-07-31           2011               7
52726    2011-03-10           2011               3


## 8. Feature Engineering

### 8.1 Water-Point Age

The `construction_year` variable and the newly derived `recorded_year` variable are used to calculate the approximate age of each water point at the time of observation.

The engineered feature is calculated as:

`waterpoint_age = recorded_year - construction_year`

Where the construction year is unavailable, the resulting age remains missing rather than assigning an artificial value.

In [37]:
X_train["waterpoint_age"] = (
    X_train["recorded_year"] - X_train["construction_year"]
)

X_test["waterpoint_age"] = (
    X_test["recorded_year"] - X_test["construction_year"]
)

print("Water-point age summary:")
print(X_train["waterpoint_age"].describe())

Water-point age summary:
count    31003.000000
mean        15.343289
std         12.524152
min         -7.000000
25%          4.000000
50%         13.000000
75%         25.000000
max         53.000000
Name: waterpoint_age, dtype: float64


In [38]:
print(
    "Negative ages - training:",
    (X_train["waterpoint_age"] < 0).sum()
)

print(
    "Negative ages - testing:",
    (X_test["waterpoint_age"] < 0).sum()
)

print(
    "Missing ages - training:",
    X_train["waterpoint_age"].isna().sum()
)

print(
    "\nExample records:"
)

print(
    X_train[
        [
            "construction_year",
            "recorded_year",
            "waterpoint_age"
        ]
    ].head(10)
)

Negative ages - training: 9
Negative ages - testing: 0
Missing ages - training: 16517

Example records:
       construction_year  recorded_year  waterpoint_age
43360                NaN           2011             NaN
7263              2008.0           2011             3.0
2486              2010.0           2011             1.0
313                  NaN           2011             NaN
52726                NaN           2011             NaN
8558              1986.0           2011            25.0
2559              1995.0           2013            18.0
54735                NaN           2011             NaN
25763                NaN           2011             NaN
44540                NaN           2011             NaN


### 8.2 Investigation of Negative Water-Point Ages

A small number of training observations produce negative `waterpoint_age` values.

A negative age indicates that the recorded construction year occurs after the observation year, which is inconsistent with the interpretation of water-point age.

These observations are investigated before deciding how to treat them. The original records are retained unless there is sufficient evidence that only the affected construction-year values are invalid.

In [39]:
negative_age_train = X_train["waterpoint_age"] < 0

print(
    X_train.loc[
        negative_age_train,
        [
            "construction_year",
            "recorded_year",
            "waterpoint_age"
        ]
    ]
)

       construction_year  recorded_year  waterpoint_age
27501             2009.0           2004            -5.0
39559             2009.0           2004            -5.0
8729              2008.0           2004            -4.0
33942             2007.0           2004            -3.0
23373             2009.0           2004            -5.0
13366             2011.0           2004            -7.0
48555             2008.0           2004            -4.0
10441             2006.0           2004            -2.0
32619             2005.0           2004            -1.0


### 8.3 Treatment of Inconsistent Construction Years

The investigation identified a small number of observations where `construction_year` is later than `recorded_year`, producing a negative water-point age.

Because the inconsistency is specific to the construction-year value, the affected observations are retained while the inconsistent `construction_year` values are treated as missing.

This avoids unnecessary loss of complete water-point records while preventing logically impossible ages from entering the modelling process.

In [40]:
invalid_age_train = (
    X_train["construction_year"] > X_train["recorded_year"]
)

invalid_age_test = (
    X_test["construction_year"] > X_test["recorded_year"]
)

print(
    "Inconsistent construction years - training:",
    invalid_age_train.sum()
)

print(
    "Inconsistent construction years - testing:",
    invalid_age_test.sum()
)

Inconsistent construction years - training: 9
Inconsistent construction years - testing: 0


In [41]:
X_train.loc[
    invalid_age_train,
    "construction_year"
] = np.nan

X_test.loc[
    invalid_age_test,
    "construction_year"
] = np.nan

In [42]:
X_train["waterpoint_age"] = (
    X_train["recorded_year"] - X_train["construction_year"]
)

X_test["waterpoint_age"] = (
    X_test["recorded_year"] - X_test["construction_year"]
)

In [43]:
print(
    "Negative ages - training:",
    (X_train["waterpoint_age"] < 0).sum()
)

print(
    "Negative ages - testing:",
    (X_test["waterpoint_age"] < 0).sum()
)

print(
    "Missing ages - training:",
    X_train["waterpoint_age"].isna().sum()
)

Negative ages - training: 0
Negative ages - testing: 0
Missing ages - training: 16526


## 8.4 Remove the Original Date Variable

After extracting the relevant temporal components from `date_recorded`, the original date variable is no longer required as a raw predictor.

The original date column is removed to prevent it from being treated as a high-cardinality categorical variable during later preprocessing.

In [44]:
X_train = X_train.drop(columns=["date_recorded"])
X_test = X_test.drop(columns=["date_recorded"])

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (47520, 40)
Testing shape: (11880, 40)


## 9. High-Cardinality Categorical Feature Treatment

The remaining high-cardinality categorical variables are examined before encoding.

Rather than removing variables solely because they have many categories, category-frequency patterns are considered alongside missingness and the semantic meaning of each variable.

Rare categories may be grouped into an `"Other"` category before encoding to reduce sparsity and improve the generalisability of the resulting feature representation.

In [45]:
remaining_high_cardinality = [
    "subvillage",
    "scheme_name",
    "ward",
    "installer",
    "funder"
]

rare_category_summary = []

for col in remaining_high_cardinality:
    counts = X_train[col].value_counts(dropna=False)

    rare_category_summary.append({
        "feature": col,
        "unique_categories": counts.size,
        "categories_1_or_less": (counts == 1).sum(),
        "categories_5_or_less": (counts <= 5).sum(),
        "categories_10_or_less": (counts <= 10).sum(),
        "missing_count": X_train[col].isna().sum(),
        "missing_percentage": X_train[col].isna().mean() * 100
    })

rare_category_summary = pd.DataFrame(rare_category_summary)

rare_category_summary.round(2)

,feature,unique_categories,categories_1_or_less,categories_5_or_less,categories_10_or_less,missing_count,missing_percentage
0,subvillage,17232,8948,15838,16815,286,0.60
1,scheme_name,2563,709,1561,1985,23050,48.51
2,ward,2082,53,316,692,0,0.00
3,installer,1930,971,1436,1577,2917,6.14
4,funder,1716,868,1251,1358,2906,6.12


In [46]:
for col in remaining_high_cardinality:
    print(f"\n--- {col} ---")
    print(X_train[col].value_counts(dropna=False).head(10))


--- subvillage ---
subvillage
Shuleni     420
Majengo     409
Madukani    404
Kati        306
NaN         286
Mtakuja     211
Sokoni      191
M           150
Mbuyuni     132
Muungano    130
Name: count, dtype: int64

--- scheme_name ---
scheme_name
NaN                            23050
K                                548
Borehole                         435
Chalinze wate                    325
M                                322
DANIDA                           310
Government                       251
Ngana water supplied scheme      211
wanging'ombe water supply s      208
wanging'ombe supply scheme       188
Name: count, dtype: int64

--- ward ---
ward
Igosi           257
Imalinyi        209
Siha Kati       189
Mdandu          178
Nduruma         176
Mishamo         164
Chalinze        163
Msindo          157
Maji ya Chai    154
Kitunda         153
Name: count, dtype: int64

--- installer ---
installer
DWE           13978
NaN            2917
Government     1458
RWE             960


### 9.1 Investigation of Potential Placeholder Categories

Categorical variables are checked for placeholder-like values such as `"0"`, empty strings, or other non-informative representations.

These values are investigated before encoding because treating an invalid placeholder as a genuine category could introduce misleading information into the model.

No values are modified until their frequency and distribution have been examined.

In [47]:
placeholder_values = ["0", "", "unknown", "Unknown", "none", "None", "-"]

for col in X_train.select_dtypes(include="object").columns:
    found = {}

    for value in placeholder_values:
        count = (X_train[col] == value).sum()
        if count > 0:
            found[value] = count

    if found:
        print(f"\n--- {col} ---")
        print(found)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_8712\1825824048.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in X_train.select_dtypes(include="object").columns:



--- funder ---
{'0': np.int64(607), 'Unknown': np.int64(3)}

--- installer ---
{'0': np.int64(607), 'unknown': np.int64(1), 'Unknown': np.int64(2), '-': np.int64(3)}

--- wpt_name ---
{'Unknown': np.int64(3), 'none': np.int64(2879)}

--- scheme_name ---
{'none': np.int64(21)}

--- management ---
{'unknown': np.int64(453)}

--- management_group ---
{'unknown': np.int64(453)}

--- payment ---
{'unknown': np.int64(6532)}

--- payment_type ---
{'unknown': np.int64(6532)}

--- water_quality ---
{'unknown': np.int64(1503)}

--- quality_group ---
{'unknown': np.int64(1503)}

--- quantity ---
{'unknown': np.int64(635)}

--- quantity_group ---
{'unknown': np.int64(635)}

--- source ---
{'unknown': np.int64(59)}

--- source_class ---
{'unknown': np.int64(234)}


### 9.1 Standardisation of Placeholder Values

The categorical variables contain several representations of missing or unspecified information, including values such as `"unknown"`, `"Unknown"`, `"none"`, `"None"`, `"-"`, and `"0"`.

Treating these representations as separate categorical levels could incorrectly imply that they represent meaningful groups.

Therefore, identified placeholder values are standardised as missing values before categorical imputation and encoding.

This step is performed separately on the training and testing predictors and does not use the target variable.

In [48]:
placeholder_map = {
    "0": np.nan,
    "": np.nan,
    "unknown": np.nan,
    "Unknown": np.nan,
    "none": np.nan,
    "None": np.nan,
    "-": np.nan
}

categorical_cols = X_train.select_dtypes(include="object").columns

X_train[categorical_cols] = X_train[categorical_cols].replace(
    placeholder_map
)

X_test[categorical_cols] = X_test[categorical_cols].replace(
    placeholder_map
)

print("Placeholder standardisation completed.")

C:\Users\ASUS\AppData\Local\Temp\ipykernel_8712\152843942.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include="object").columns


Placeholder standardisation completed.


In [49]:
for col in ["installer", "funder"]:
    print(f"\n--- {col} ---")
    print(
        X_train[col]
        .value_counts(dropna=False)
        .head(10)
    )


--- installer ---
installer
DWE                   13978
NaN                    3530
Government             1458
RWE                     960
Commu                   846
DANIDA                  842
KKKT                    709
Hesawa                  668
TCRS                    576
Central government      489
Name: count, dtype: int64

--- funder ---
funder
Government Of Tanzania    7321
NaN                       3516
Danida                    2491
Hesawa                    1760
Rwssp                     1107
World Bank                1058
Kkkt                      1019
World Vision              1001
Unicef                     840
Tasaf                      719
Name: count, dtype: int64


In [50]:
categorical_missing_after_placeholder = pd.DataFrame({
    "missing_count": X_train[categorical_cols].isna().sum(),
    "missing_percentage": (
        X_train[categorical_cols].isna().mean() * 100
    )
})

categorical_missing_after_placeholder = (
    categorical_missing_after_placeholder
    .query("missing_count > 0")
    .sort_values("missing_percentage", ascending=False)
)

categorical_missing_after_placeholder.round(2)

,missing_count,missing_percentage
scheme_name,23071,48.55
payment,6532,13.75
payment_type,6532,13.75
installer,3530,7.43
funder,3516,7.40
scheme_management,3129,6.58
wpt_name,2884,6.07
public_meeting,2644,5.56
permit,2443,5.14
water_quality,1503,3.16


In [51]:
X_train = X_train.drop(columns=["wpt_name"], errors="ignore")
X_test = X_test.drop(columns=["wpt_name"], errors="ignore")

print("wpt_name in training:", "wpt_name" in X_train.columns)
print("wpt_name in testing:", "wpt_name" in X_test.columns)

wpt_name in training: False
wpt_name in testing: False


## 10. Feature Redundancy and Hierarchical Variable Investigation

Several categorical variables represent related information at different levels of detail.

Examples include `payment` and `payment_type`, `water_quality` and `quality_group`, `quantity` and `quantity_group`, `management` and `management_group`, and `source` and `source_class`.

These relationships are investigated before encoding to determine whether the grouped variables provide additional information or simply duplicate information already represented by the more detailed variables.

The investigation is performed using the training data only.

In [52]:
categorical_cols = X_train.select_dtypes(include="object").columns.tolist()

print("Number of categorical features:", len(categorical_cols))
print("\nCategorical features:")
print(categorical_cols)

print("\nChecking wpt_name:")
print("wpt_name in X_train:", "wpt_name" in X_train.columns)
print("wpt_name in X_test:", "wpt_name" in X_test.columns)

Number of categorical features: 27

Categorical features:
['funder', 'installer', 'basin', 'subvillage', 'region', 'lga', 'ward', 'public_meeting', 'scheme_management', 'scheme_name', 'permit', 'extraction_type', 'extraction_type_group', 'extraction_type_class', 'management', 'management_group', 'payment', 'payment_type', 'water_quality', 'quality_group', 'quantity', 'quantity_group', 'source', 'source_type', 'source_class', 'waterpoint_type', 'waterpoint_type_group']

Checking wpt_name:
wpt_name in X_train: False
wpt_name in X_test: False


C:\Users\ASUS\AppData\Local\Temp\ipykernel_8712\2006517678.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include="object").columns.tolist()


In [53]:
categorical_missing = pd.DataFrame({
    "missing_count": X_train[categorical_cols].isna().sum(),
    "missing_percentage": (
        X_train[categorical_cols].isna().mean() * 100
    )
})

categorical_missing = (
    categorical_missing[
        categorical_missing["missing_count"] > 0
    ]
    .sort_values("missing_percentage", ascending=False)
)

categorical_missing.round(2)

,missing_count,missing_percentage
scheme_name,23071,48.55
payment_type,6532,13.75
payment,6532,13.75
installer,3530,7.43
funder,3516,7.40
scheme_management,3129,6.58
public_meeting,2644,5.56
permit,2443,5.14
water_quality,1503,3.16
quality_group,1503,3.16


In [54]:
feature_pairs = [
    ("payment", "payment_type"),
    ("water_quality", "quality_group"),
    ("quantity", "quantity_group"),
    ("management", "management_group"),
    ("source", "source_class")
]

for detailed, grouped in feature_pairs:

    print(f"\n===== {detailed} → {grouped} =====")

    mapping_check = (
        X_train
        .groupby(detailed, dropna=False)[grouped]
        .nunique(dropna=False)
        .sort_values(ascending=False)
    )

    print(mapping_check.head(20))


===== payment → payment_type =====
payment
never pay                1
other                    1
pay annually             1
pay monthly              1
pay per bucket           1
pay when scheme fails    1
NaN                      1
Name: payment_type, dtype: int64

===== water_quality → quality_group =====
water_quality
coloured              1
fluoride              1
fluoride abandoned    1
milky                 1
salty                 1
salty abandoned       1
soft                  1
NaN                   1
Name: quality_group, dtype: int64

===== quantity → quantity_group =====
quantity
dry             1
enough          1
insufficient    1
seasonal        1
NaN             1
Name: quantity_group, dtype: int64

===== management → management_group =====
management
company             1
other               1
other - school      1
parastatal          1
private operator    1
trust               1
vwc                 1
water authority     1
water board         1
wua                 1
wug 

In [55]:
for detailed, grouped in feature_pairs:

    print(f"\n===== {grouped} → {detailed} =====")

    reverse_check = (
        X_train
        .groupby(grouped, dropna=False)[detailed]
        .nunique(dropna=False)
        .sort_values(ascending=False)
    )

    print(reverse_check.head(20))


===== payment_type → payment =====
payment_type
annually      1
monthly       1
never pay     1
on failure    1
other         1
per bucket    1
NaN           1
Name: payment, dtype: int64

===== quality_group → water_quality =====
quality_group
fluoride    2
salty       2
colored     1
good        1
milky       1
NaN         1
Name: water_quality, dtype: int64

===== quantity_group → quantity =====
quantity_group
dry             1
enough          1
insufficient    1
seasonal        1
NaN             1
Name: quantity, dtype: int64

===== management_group → management =====
management_group
commercial    4
user-group    4
other         2
parastatal    1
NaN           1
Name: management, dtype: int64

===== source_class → source =====
source_class
groundwater    4
surface        4
NaN            2
Name: source, dtype: int64


## 10.1 Feature Redundancy Decisions

The relationship analysis identified several categorical variables that represent the same underlying concepts at different levels of detail.

`payment` and `payment_type` showed a one-to-one relationship in both directions, indicating that they provide duplicate information. Therefore, `payment` is removed and `payment_type` is retained.

`quantity` and `quantity_group` also showed a one-to-one relationship. Therefore, `quantity_group` is removed while `quantity` is retained.

For `water_quality`, `management`, and `source`, the corresponding grouped variables contained fewer categories than the detailed variables. The reverse mapping showed multiple detailed categories within some grouped categories. Therefore, the detailed variables are retained and the broader grouping variables are removed.

This reduces redundant predictors while preserving the more detailed information available in the original dataset.

In [56]:
redundant_features = [
    "payment",
    "quantity_group",
    "quality_group",
    "management_group",
    "source_class"
]

X_train = X_train.drop(
    columns=redundant_features,
    errors="ignore"
)

X_test = X_test.drop(
    columns=redundant_features,
    errors="ignore"
)

print("Removed features:")
print(redundant_features)

print("\nTraining shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Removed features:
['payment', 'quantity_group', 'quality_group', 'management_group', 'source_class']

Training shape: (47520, 34)
Testing shape: (11880, 34)


In [57]:
print("Remaining redundant features:")

for col in redundant_features:
    print(
        col,
        "→",
        "present" if col in X_train.columns else "removed"
    )

Remaining redundant features:
payment → removed
quantity_group → removed
quality_group → removed
management_group → removed
source_class → removed


## 11. Numerical Feature Quality Analysis

The remaining numerical variables are investigated before imputation and scaling.

The analysis considers the variable meaning, missingness, value ranges, unusual observations, and potential outliers.

Numeric columns representing administrative or geographic codes are treated according to their meaning rather than automatically being considered continuous measurements.

The results will be used to determine appropriate imputation, transformation, and scaling strategies.

11.1 Identify the current numerical features

In [58]:
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()

print("Number of numeric columns:", len(numeric_cols))
print("\nNumeric columns:")
print(numeric_cols)

Number of numeric columns: 12

Numeric columns:
['amount_tsh', 'gps_height', 'longitude', 'latitude', 'num_private', 'region_code', 'district_code', 'population', 'construction_year', 'recorded_year', 'recorded_month', 'waterpoint_age']


Step 11.2 — Separate numeric measurements from numeric codes

In [59]:
numeric_code_features = [
    "region_code",
    "district_code"
]

numeric_measurements = [
    col for col in numeric_cols
    if col not in numeric_code_features
]

print("Numeric code features:")
print(numeric_code_features)

print("\nNumeric measurement features:")
print(numeric_measurements)

Numeric code features:
['region_code', 'district_code']

Numeric measurement features:
['amount_tsh', 'gps_height', 'longitude', 'latitude', 'num_private', 'population', 'construction_year', 'recorded_year', 'recorded_month', 'waterpoint_age']


Step 11.3 — Missing-value analysis

In [60]:
numeric_missing = pd.DataFrame({
    "missing_count": X_train[numeric_measurements].isna().sum(),
    "missing_percentage": (
        X_train[numeric_measurements].isna().mean() * 100
    )
})

numeric_missing = (
    numeric_missing[
        numeric_missing["missing_count"] > 0
    ]
    .sort_values("missing_percentage", ascending=False)
)

numeric_missing.round(2)

,missing_count,missing_percentage
waterpoint_age,16526,34.78
construction_year,16526,34.78
latitude,1442,3.03
longitude,1442,3.03


Step 11.4 — Numerical summary

In [61]:
numeric_summary = X_train[numeric_measurements].describe().T

numeric_summary

,count,mean,std,min,25%,50%,75%,max
amount_tsh,47520.0,321.925261,3197.240487,0.000000,0.000000,0.000000,25.000000,350000.000000
gps_height,47520.0,669.567656,693.005745,-63.000000,0.000000,372.500000,1320.000000,2770.000000
longitude,46078.0,35.149033,2.604241,29.607122,33.284679,35.008578,37.223501,40.344301
latitude,46078.0,-5.884512,2.805599,-11.649440,-8.633876,-5.170151,-3.375068,-0.998464
num_private,47520.0,0.477736,13.312977,0.000000,0.000000,0.000000,0.000000,1776.000000
population,47520.0,179.805513,463.081564,0.000000,0.000000,25.000000,215.000000,15300.000000
construction_year,30994.0,1996.822224,12.499577,1960.000000,1988.000000,2000.000000,2008.000000,2013.000000
recorded_year,47520.0,2011.921907,0.959690,2002.000000,2011.000000,2012.000000,2013.000000,2013.000000
recorded_month,47520.0,4.366604,3.025041,1.000000,2.000000,3.000000,7.000000,12.000000
waterpoint_age,30994.0,15.348906,12.521597,0.000000,4.000000,13.000000,25.000000,53.000000


Step 11.5 — Check for suspicious numerical values

In [62]:
for col in numeric_measurements:
    print(f"\n===== {col} =====")
    print("Minimum:", X_train[col].min())
    print("Maximum:", X_train[col].max())
    print("Mean:", round(X_train[col].mean(), 2))
    print("Median:", X_train[col].median())


===== amount_tsh =====
Minimum: 0.0
Maximum: 350000.0
Mean: 321.93
Median: 0.0

===== gps_height =====
Minimum: -63
Maximum: 2770
Mean: 669.57
Median: 372.5

===== longitude =====
Minimum: 29.6071219
Maximum: 40.34430089
Mean: 35.15
Median: 35.008578455

===== latitude =====
Minimum: -11.64944018
Maximum: -0.99846435
Mean: -5.88
Median: -5.17015135

===== num_private =====
Minimum: 0
Maximum: 1776
Mean: 0.48
Median: 0.0

===== population =====
Minimum: 0
Maximum: 15300
Mean: 179.81
Median: 25.0

===== construction_year =====
Minimum: 1960.0
Maximum: 2013.0
Mean: 1996.82
Median: 2000.0

===== recorded_year =====
Minimum: 2002
Maximum: 2013
Mean: 2011.92
Median: 2012.0

===== recorded_month =====
Minimum: 1
Maximum: 12
Mean: 4.37
Median: 3.0

===== waterpoint_age =====
Minimum: 0.0
Maximum: 53.0
Mean: 15.35
Median: 13.0


Step 11.6 — Outlier investigation using IQR

In [63]:
outlier_summary = []

for col in numeric_measurements:

    series = X_train[col].dropna()

    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = ((series < lower) | (series > upper)).sum()

    outlier_summary.append({
        "feature": col,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "lower_bound": lower,
        "upper_bound": upper,
        "outlier_count": outliers,
        "outlier_percentage": outliers / len(series) * 100
    })

outlier_summary = pd.DataFrame(outlier_summary)

outlier_summary.sort_values(
    "outlier_percentage",
    ascending=False
).round(2)

,feature,Q1,Q3,IQR,lower_bound,upper_bound,outlier_count,outlier_percentage
0,amount_tsh,0.00,25.00,25.00,-37.50,62.50,8970,18.88
5,population,0.00,215.00,215.00,-322.50,537.50,3451,7.26
4,num_private,0.00,0.00,0.00,0.00,0.00,588,1.24
7,recorded_year,2011.00,2013.00,2.00,2008.00,2016.00,26,0.05
1,gps_height,0.00,1320.00,1320.00,-1980.00,3300.00,0,0.00
3,latitude,-8.63,-3.38,5.26,-16.52,4.51,0,0.00
2,longitude,33.28,37.22,3.94,27.38,43.13,0,0.00
6,construction_year,1988.00,2008.00,20.00,1958.00,2038.00,0,0.00
8,recorded_month,2.00,7.00,5.00,-5.50,14.50,0,0.00
9,waterpoint_age,4.00,25.00,21.00,-27.50,56.50,0,0.00


Step 11.7 — Check construction_year

In [64]:
print("Construction year missing:",
      X_train["construction_year"].isna().sum())

print("\nConstruction year range:")

print(
    X_train["construction_year"]
    .dropna()
    .agg(["min", "max"])
)

print("\nMost common construction years:")

print(
    X_train["construction_year"]
    .value_counts()
    .sort_index()
    .head(20)
)

Construction year missing: 16526

Construction year range:
min    1960.0
max    2013.0
Name: construction_year, dtype: float64

Most common construction years:
construction_year
1960.0     82
1961.0     18
1962.0     22
1963.0     73
1964.0     35
1965.0     16
1966.0     15
1967.0     71
1968.0     63
1969.0     48
1970.0    329
1971.0    111
1972.0    581
1973.0    150
1974.0    562
1975.0    349
1976.0    335
1977.0    163
1978.0    817
1979.0    152
Name: count, dtype: int64


## 12. Numerical Feature Preprocessing Decisions

The numerical analysis was used to determine appropriate preprocessing strategies for each numerical feature.

The preprocessing decisions consider missingness, variable meaning, plausible ranges, skewness, and the distinction between genuine extreme observations and invalid values.

Extreme values identified using the IQR rule are not automatically removed because unusually large values may represent genuine characteristics of rural water points rather than data-entry errors.

Missing values will be handled using a leakage-free approach, where imputation parameters are learned from the training data only and then applied to the testing data.

### 12.1 Treatment of Region and District Codes

`region_code` and `district_code` are numerical in storage format, but their values represent geographic categories rather than continuous measurements.

Therefore, treating these variables as ordinary continuous numerical measurements would incorrectly imply that the difference between category codes has a quantitative meaning.

They will therefore be treated as categorical variables during the later encoding stage.

In [65]:
numeric_code_features = ["region_code", "district_code"]

# Remove them from the numerical measurement list
numeric_measurements = [
    col for col in numeric_measurements
    if col not in numeric_code_features
]

print("Numerical measurement features:")
print(numeric_measurements)

print("\nNumerical code features:")
print(numeric_code_features)

Numerical measurement features:
['amount_tsh', 'gps_height', 'longitude', 'latitude', 'num_private', 'population', 'construction_year', 'recorded_year', 'recorded_month', 'waterpoint_age']

Numerical code features:
['region_code', 'district_code']


### 12.2 Treatment of Construction Year and Water-Point Age

`waterpoint_age` was engineered from `recorded_year - construction_year`.

The analysis shows that `construction_year` and `waterpoint_age` have the same missingness pattern, with 16,526 missing observations in the training data.

Keeping both variables would therefore introduce redundant information.

A missingness indicator for `construction_year` is retained to preserve information about whether the original construction year was unavailable.

The original `construction_year` variable will then be removed, while `waterpoint_age` will be retained as the engineered temporal feature.

In [66]:
# Preserve whether construction year was originally missing
X_train["construction_year_missing"] = X_train["construction_year"].isna().astype(int)
X_test["construction_year_missing"] = X_test["construction_year"].isna().astype(int)

print("Construction-year missing indicator:")
print(X_train["construction_year_missing"].value_counts())

print("\nTraining missing count:")
print(X_train["construction_year_missing"].sum())

print("Testing missing count:")
print(X_test["construction_year_missing"].sum())

Construction-year missing indicator:
construction_year_missing
0    30994
1    16526
Name: count, dtype: int64

Training missing count:
16526
Testing missing count:
4192


The median used for imputing missing water-point ages is calculated from the training dataset only. The same training-derived median is then applied to both training and testing data. This prevents information from the testing dataset from influencing the preprocessing decision.

In [67]:
X_train = X_train.drop(columns=["construction_year"], errors="ignore")
X_test = X_test.drop(columns=["construction_year"], errors="ignore")

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (47520, 34)
Testing shape: (11880, 34)


Step 12.3 — What about the missing waterpoint_age?

In [68]:
print("Water-point age median:", X_train["waterpoint_age"].median())
print("Water-point age mean:", X_train["waterpoint_age"].mean())

Water-point age median: 13.0
Water-point age mean: 15.348906239917403


In [69]:
age_median = X_train["waterpoint_age"].median()

X_train["waterpoint_age"] = X_train["waterpoint_age"].fillna(age_median)
X_test["waterpoint_age"] = X_test["waterpoint_age"].fillna(age_median)

print("Remaining missing waterpoint_age - training:",
      X_train["waterpoint_age"].isna().sum())

print("Remaining missing waterpoint_age - testing:",
      X_test["waterpoint_age"].isna().sum())

Remaining missing waterpoint_age - training: 0
Remaining missing waterpoint_age - testing: 0


### 12.3 Treatment of Missing Geographic Coordinates

Latitude and longitude contain approximately 3.03% missing values in the training data.

Because the proportion of missing values is relatively small, the observations will be retained rather than removed.

The missing coordinate values will be imputed using the corresponding training-set median.

The imputation values are calculated using the training data only to prevent information leakage.

In [70]:
coordinate_features = ["latitude", "longitude"]

coordinate_medians = X_train[coordinate_features].median()

print("Training coordinate medians:")
print(coordinate_medians)

Training coordinate medians:
latitude     -5.170151
longitude    35.008578
dtype: float64


In [71]:
X_train[coordinate_features] = X_train[coordinate_features].fillna(
    coordinate_medians
)

X_test[coordinate_features] = X_test[coordinate_features].fillna(
    coordinate_medians
)

print("Remaining missing coordinates - training:")
print(X_train[coordinate_features].isna().sum())

print("\nRemaining missing coordinates - testing:")
print(X_test[coordinate_features].isna().sum())

Remaining missing coordinates - training:
latitude     0
longitude    0
dtype: int64

Remaining missing coordinates - testing:
latitude     0
longitude    0
dtype: int64


### 12.4 Treatment of Extreme Numerical Values

The IQR analysis identified a relatively large number of observations
for `amount_tsh` and `population` as statistical outliers.

These observations are not automatically removed because an IQR-based
outlier does not necessarily represent an invalid observation.

For example, `amount_tsh` is highly concentrated around zero but also
contains larger values. These values may represent genuine differences
in water-point characteristics.

Therefore, the observations are retained rather than deleted.
The outlier analysis is used to avoid removing potentially informative
water-point records.

In [72]:
numeric_code_features = ["region_code", "district_code"]

numeric_measurements = [
    col for col in numeric_measurements
    if col not in numeric_code_features
]

print(numeric_measurements)

['amount_tsh', 'gps_height', 'longitude', 'latitude', 'num_private', 'population', 'construction_year', 'recorded_year', 'recorded_month', 'waterpoint_age']


In [73]:
age_median = X_train["waterpoint_age"].median()

print("Training median waterpoint age:", age_median)

X_train["waterpoint_age"] = X_train["waterpoint_age"].fillna(age_median)
X_test["waterpoint_age"] = X_test["waterpoint_age"].fillna(age_median)

print("\nRemaining missing waterpoint_age - training:",
      X_train["waterpoint_age"].isna().sum())

print("Remaining missing waterpoint_age - testing:",
      X_test["waterpoint_age"].isna().sum())

Training median waterpoint age: 13.0

Remaining missing waterpoint_age - training: 0
Remaining missing waterpoint_age - testing: 0


In [74]:
coordinate_features = ["latitude", "longitude"]

coordinate_medians = X_train[coordinate_features].median()

X_train[coordinate_features] = X_train[coordinate_features].fillna(coordinate_medians)
X_test[coordinate_features] = X_test[coordinate_features].fillna(coordinate_medians)

In [75]:
print("Remaining numerical missing values - training:")

print(
    X_train.select_dtypes(include=["number"])
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("\nRemaining numerical missing values - testing:")

print(
    X_test.select_dtypes(include=["number"])
    .isna()
    .sum()
    .sort_values(ascending=False)
)

Remaining numerical missing values - training:
amount_tsh                   0
gps_height                   0
longitude                    0
latitude                     0
num_private                  0
region_code                  0
district_code                0
population                   0
recorded_year                0
recorded_month               0
waterpoint_age               0
construction_year_missing    0
dtype: int64

Remaining numerical missing values - testing:
amount_tsh                   0
gps_height                   0
longitude                    0
latitude                     0
num_private                  0
region_code                  0
district_code                0
population                   0
recorded_year                0
recorded_month               0
waterpoint_age               0
construction_year_missing    0
dtype: int64


In [76]:
categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()

categorical_missing = pd.DataFrame({
    "missing_count": X_train[categorical_cols].isna().sum(),
    "missing_percentage": X_train[categorical_cols].isna().mean() * 100
})

categorical_missing = (
    categorical_missing[categorical_missing["missing_count"] > 0]
    .sort_values("missing_percentage", ascending=False)
)

categorical_missing.round(2)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_8712\3324639424.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()


,missing_count,missing_percentage
scheme_name,23071,48.55
payment_type,6532,13.75
installer,3530,7.43
funder,3516,7.40
scheme_management,3129,6.58
public_meeting,2644,5.56
permit,2443,5.14
water_quality,1503,3.16
quantity,635,1.34
management,453,0.95


Treatment of Missing Categorical Values

The categorical features contain missing values, with `scheme_name` having the highest proportion of missing observations (48.55%). Rather than replacing missing categorical values with the most frequent category, missing values are treated as a separate category labelled `"Missing"`.

This approach preserves the information that a value was not recorded and avoids assigning missing observations to an existing category without evidence.

The treatment is applied separately to the training and testing predictor datasets and does not use the target variable.

In [77]:
for col in categorical_cols:
    X_train[col] = X_train[col].fillna("Missing")
    X_test[col] = X_test[col].fillna("Missing")

print("Remaining categorical missing values - training:")
print(X_train[categorical_cols].isna().sum().sum())

print("\nRemaining categorical missing values - testing:")
print(X_test[categorical_cols].isna().sum().sum())

Remaining categorical missing values - training:
0

Remaining categorical missing values - testing:
0


### 12.5 Categorical Feature Cardinality and Encoding Strategy

Categorical features contain substantially different numbers of unique categories. 
Low-cardinality variables can be encoded using one-hot encoding, whereas high-cardinality 
variables require additional treatment to avoid producing an excessively large number of 
encoded features.

Therefore, the remaining categorical variables are examined according to their cardinality 
before selecting the final encoding strategy.

In [78]:
# Recalculate categorical features after preprocessing
categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()

categorical_cardinality = pd.DataFrame({
    "unique_values": X_train[categorical_cols].nunique(),
    "unique_ratio_percent": (
        X_train[categorical_cols].nunique() / len(X_train) * 100
    )
}).sort_values("unique_values", ascending=False)

categorical_cardinality.round(2)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_8712\1536388897.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()


,unique_values,unique_ratio_percent
subvillage,17232,36.26
scheme_name,2562,5.39
ward,2082,4.38
installer,1926,4.05
funder,1714,3.61
lga,124,0.26
region,21,0.04
extraction_type,18,0.04
extraction_type_group,13,0.03
scheme_management,12,0.03


### 12.6 Classification of Categorical Features by Cardinality

The categorical variables are classified according to the number of unique categories they contain. Low-cardinality variables can be represented using one-hot encoding without producing a large number of additional features. High-cardinality variables require a more compact representation to avoid excessive dimensionality.

The cardinality analysis is therefore used to determine which variables can be one-hot encoded directly and which variables require alternative encoding.

In [79]:
# Define cardinality groups

low_cardinality = [
    col for col in categorical_cols
    if X_train[col].nunique() <= 20
]

high_cardinality = [
    col for col in categorical_cols
    if X_train[col].nunique() > 20
]

print("Low-cardinality categorical features:")
print(low_cardinality)

print("\nHigh-cardinality categorical features:")
print(high_cardinality)

print("\nNumber of low-cardinality features:", len(low_cardinality))
print("Number of high-cardinality features:", len(high_cardinality))

Low-cardinality categorical features:
['basin', 'public_meeting', 'scheme_management', 'permit', 'extraction_type', 'extraction_type_group', 'extraction_type_class', 'management', 'payment_type', 'water_quality', 'quantity', 'source', 'source_type', 'waterpoint_type', 'waterpoint_type_group']

High-cardinality categorical features:
['funder', 'installer', 'subvillage', 'region', 'lga', 'ward', 'scheme_name']

Number of low-cardinality features: 15
Number of high-cardinality features: 7


### 12.7 Treatment of High-Cardinality Categorical Features

Several categorical variables contain a large number of unique categories, particularly
`subvillage`, `scheme_name`, `ward`, `installer`, and `funder`. Direct one-hot encoding
of these variables would produce a very large number of additional predictors.

To reduce dimensionality while retaining information about category occurrence, frequency
encoding is applied to the high-cardinality variables. The frequency of each category is
calculated using the training data only. The resulting frequency values are then applied
to both the training and testing datasets.

This approach avoids using the target variable and prevents target leakage during
preprocessing.

In [80]:
# Features selected for frequency encoding
frequency_encode_cols = [
    "funder",
    "installer",
    "subvillage",
    "ward",
    "scheme_name"
]

print("Features selected for frequency encoding:")
print(frequency_encode_cols)

Features selected for frequency encoding:
['funder', 'installer', 'subvillage', 'ward', 'scheme_name']


In [81]:
# Apply frequency encoding using training data only

frequency_maps = {}

for col in frequency_encode_cols:
    frequency = X_train[col].value_counts(normalize=True)

    frequency_maps[col] = frequency

    X_train[col + "_freq"] = X_train[col].map(frequency)
    X_test[col + "_freq"] = X_test[col].map(frequency)

    # Unseen categories in testing receive frequency 0
    X_test[col + "_freq"] = X_test[col + "_freq"].fillna(0)

print("Frequency encoding completed.")

Frequency encoding completed.


In [82]:
for col in frequency_encode_cols:
    print(
        f"{col}:",
        "train min =", X_train[col + "_freq"].min(),
        "train max =", X_train[col + "_freq"].max(),
        "test min =", X_test[col + "_freq"].min(),
        "test max =", X_test[col + "_freq"].max()
    )

funder: train min = 2.1043771043771045e-05 train max = 0.15406144781144782 test min = 0.0 test max = 0.15406144781144782
installer: train min = 2.1043771043771045e-05 train max = 0.29414983164983166 test min = 0.0 test max = 0.29414983164983166
subvillage: train min = 2.1043771043771045e-05 train max = 0.008838383838383838 test min = 0.0 test max = 0.008838383838383838
ward: train min = 2.1043771043771045e-05 train max = 0.005408249158249158 test min = 0.0 test max = 0.005408249158249158
scheme_name: train min = 2.1043771043771045e-05 train max = 0.48550084175084174 test min = 0.0 test max = 0.48550084175084174


In [83]:
# Verify that frequency-encoded features contain no missing values

freq_cols = [col + "_freq" for col in frequency_encode_cols]

print("Missing values in frequency-encoded training features:")
print(X_train[freq_cols].isna().sum())

print("\nMissing values in frequency-encoded testing features:")
print(X_test[freq_cols].isna().sum())

Missing values in frequency-encoded training features:
funder_freq         0
installer_freq      0
subvillage_freq     0
ward_freq           0
scheme_name_freq    0
dtype: int64

Missing values in frequency-encoded testing features:
funder_freq         0
installer_freq      0
subvillage_freq     0
ward_freq           0
scheme_name_freq    0
dtype: int64


In [84]:
print("Frequency-encoded columns:")
print(freq_cols)

print("\nTraining shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Frequency-encoded columns:
['funder_freq', 'installer_freq', 'subvillage_freq', 'ward_freq', 'scheme_name_freq']

Training shape: (47520, 39)
Testing shape: (11880, 39)


### 12.8 Removal of Original High-Cardinality Variables

After frequency encoding, the original high-cardinality categorical variables are removed from the training and testing predictor datasets.

The frequency-encoded variables are retained as numerical representations of category occurrence. This avoids the very large dimensionality that would result from one-hot encoding the original high-cardinality variables.

The same columns are removed from both datasets to maintain an identical predictor structure.

In [85]:
# Remove original high-cardinality categorical features
# after frequency encoding

X_train = X_train.drop(
    columns=frequency_encode_cols,
    errors="ignore"
)

X_test = X_test.drop(
    columns=frequency_encode_cols,
    errors="ignore"
)

print("Removed original high-cardinality features:")
print(frequency_encode_cols)

print("\nTraining shape:", X_train.shape)
print("Testing shape:", X_test.shape)

print("\nFrequency-encoded columns retained:")
print(freq_cols)

Removed original high-cardinality features:
['funder', 'installer', 'subvillage', 'ward', 'scheme_name']

Training shape: (47520, 34)
Testing shape: (11880, 34)

Frequency-encoded columns retained:
['funder_freq', 'installer_freq', 'subvillage_freq', 'ward_freq', 'scheme_name_freq']


In [86]:
print("Original high-cardinality columns remaining:")

for col in frequency_encode_cols:
    print(
        col,
        "->",
        "present" if col in X_train.columns else "removed"
    )

Original high-cardinality columns remaining:
funder -> removed
installer -> removed
subvillage -> removed
ward -> removed
scheme_name -> removed


### 12.9 Low-Cardinality Categorical Feature Preparation

After removing the original high-cardinality variables, the remaining categorical predictors are examined to determine their number of unique categories.

Low-cardinality categorical variables can be represented using one-hot encoding without creating an excessive number of additional predictors.

The categorical features are therefore reviewed before applying the final encoding strategy.

In [87]:
# Identify remaining categorical features

categorical_cols = (
    X_train
    .select_dtypes(include=["object"])
    .columns
    .tolist()
)

print("Remaining categorical features:")
print(categorical_cols)

print("\nNumber of remaining categorical features:")
print(len(categorical_cols))

Remaining categorical features:
['basin', 'region', 'lga', 'public_meeting', 'scheme_management', 'permit', 'extraction_type', 'extraction_type_group', 'extraction_type_class', 'management', 'payment_type', 'water_quality', 'quantity', 'source', 'source_type', 'waterpoint_type', 'waterpoint_type_group']

Number of remaining categorical features:
17


C:\Users\ASUS\AppData\Local\Temp\ipykernel_8712\890610734.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  .select_dtypes(include=["object"])


In [88]:
categorical_cardinality = pd.DataFrame({
    "feature": categorical_cols,
    "unique_values": [
        X_train[col].nunique(dropna=False)
        for col in categorical_cols
    ]
})

categorical_cardinality = categorical_cardinality.sort_values(
    "unique_values"
)

categorical_cardinality

,feature,unique_values
3,public_meeting,3
5,permit,3
12,quantity,5
16,waterpoint_type_group,6
10,payment_type,7
14,source_type,7
15,waterpoint_type,7
8,extraction_type_class,7
11,water_quality,8
0,basin,9


In [89]:
one_hot_cols = [
    "public_meeting",
    "permit",
    "quantity",
    "waterpoint_type_group",
    "payment_type",
    "source_type",
    "waterpoint_type",
    "extraction_type_class",
    "water_quality",
    "basin",
    "source",
    "scheme_management",
    "management",
    "extraction_type_group",
    "extraction_type",
    "region",
    "region_code",
    "district_code"
]

high_cardinality_remaining = [
    "lga"
]

In [90]:
print("LGA cardinality:")
print(X_train["lga"].nunique())

print("\nMost common LGA categories:")
print(X_train["lga"].value_counts().head(20))

print("\nNumber of LGA categories:")
print(X_train["lga"].nunique())

print("\nRare LGA categories (5 or fewer observations):")
lga_counts = X_train["lga"].value_counts()
print((lga_counts <= 5).sum())

LGA cardinality:
124

Most common LGA categories:
lga
Njombe           2003
Moshi Rural      1005
Arusha Rural      997
Bariadi           931
Kilosa            895
Rungwe            885
Mbozi             827
Kasulu            818
Meru              815
Bagamoyo          797
Singida Rural     789
Kilombero         775
Kibondo           724
Same              693
Kyela             690
Kigoma Rural      679
Kahama            670
Magu              654
Maswa             651
Karagwe           611
Name: count, dtype: int64

Number of LGA categories:
124

Rare LGA categories (5 or fewer observations):
0


### 12.10 One-Hot Encoding of Remaining Categorical Features

After frequency encoding the high-cardinality variables, the remaining categorical predictors have relatively manageable cardinalities.

The `lga` variable contains 124 categories, but none of its categories occurs five or fewer times in the training data. Therefore, it is retained for one-hot encoding rather than frequency encoding.

The remaining categorical variables are transformed using one-hot encoding. The encoder is fitted using the training data only and then applied to both the training and testing datasets.

Unknown categories that occur only in the testing data are handled using `handle_unknown="ignore"`.

This prevents information from the testing dataset from influencing the encoding process.

In [91]:
# Remaining categorical features for one-hot encoding

one_hot_cols = [
    "public_meeting",
    "permit",
    "quantity",
    "waterpoint_type_group",
    "payment_type",
    "source_type",
    "waterpoint_type",
    "extraction_type_class",
    "water_quality",
    "basin",
    "source",
    "scheme_management",
    "management",
    "extraction_type_group",
    "extraction_type",
    "region",
    "region_code",
    "district_code",
    "lga"
]

print("Features selected for one-hot encoding:")
print(one_hot_cols)

print("\nNumber of features:")
print(len(one_hot_cols))

Features selected for one-hot encoding:
['public_meeting', 'permit', 'quantity', 'waterpoint_type_group', 'payment_type', 'source_type', 'waterpoint_type', 'extraction_type_class', 'water_quality', 'basin', 'source', 'scheme_management', 'management', 'extraction_type_group', 'extraction_type', 'region', 'region_code', 'district_code', 'lga']

Number of features:
19


In [92]:
# Check data types inside the one-hot columns

for col in one_hot_cols:
    print(f"\n{col}")
    print(X_train[col].map(type).value_counts())


public_meeting
public_meeting
<class 'bool'>    44876
<class 'str'>      2644
Name: count, dtype: int64

permit
permit
<class 'bool'>    45077
<class 'str'>      2443
Name: count, dtype: int64

quantity
quantity
<class 'str'>    47520
Name: count, dtype: int64

waterpoint_type_group
waterpoint_type_group
<class 'str'>    47520
Name: count, dtype: int64

payment_type
payment_type
<class 'str'>    47520
Name: count, dtype: int64

source_type
source_type
<class 'str'>    47520
Name: count, dtype: int64

waterpoint_type
waterpoint_type
<class 'str'>    47520
Name: count, dtype: int64

extraction_type_class
extraction_type_class
<class 'str'>    47520
Name: count, dtype: int64

water_quality
water_quality
<class 'str'>    47520
Name: count, dtype: int64

basin
basin
<class 'str'>    47520
Name: count, dtype: int64

source
source
<class 'str'>    47520
Name: count, dtype: int64

scheme_management
scheme_management
<class 'str'>    47520
Name: count, dtype: int64

management
management
<clas

In [93]:
# Convert one-hot categorical features to strings
# This ensures each column contains a uniform data type.

for col in one_hot_cols:
    X_train[col] = X_train[col].astype(str)
    X_test[col] = X_test[col].astype(str)

print("Categorical values converted to strings.")

Categorical values converted to strings.


In [94]:
from sklearn.preprocessing import OneHotEncoder

one_hot_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

one_hot_encoder.fit(X_train[one_hot_cols])

print("One-hot encoder fitted using training data.")

One-hot encoder fitted using training data.


In [95]:
X_train_encoded = one_hot_encoder.transform(
    X_train[one_hot_cols]
)

X_test_encoded = one_hot_encoder.transform(
    X_test[one_hot_cols]
)

print("Training encoded shape:", X_train_encoded.shape)
print("Testing encoded shape:", X_test_encoded.shape)

Training encoded shape: (47520, 319)
Testing encoded shape: (11880, 319)


In [97]:
one_hot_feature_names = one_hot_encoder.get_feature_names_out(one_hot_cols)

print("Number of one-hot encoded features:")
print(len(one_hot_feature_names))

print("\nFirst 20 encoded features:")
print(one_hot_feature_names[:20])

Number of one-hot encoded features:
319

First 20 encoded features:
['public_meeting_False' 'public_meeting_Missing' 'public_meeting_True'
 'permit_False' 'permit_Missing' 'permit_True' 'quantity_Missing'
 'quantity_dry' 'quantity_enough' 'quantity_insufficient'
 'quantity_seasonal' 'waterpoint_type_group_cattle trough'
 'waterpoint_type_group_communal standpipe' 'waterpoint_type_group_dam'
 'waterpoint_type_group_hand pump' 'waterpoint_type_group_improved spring'
 'waterpoint_type_group_other' 'payment_type_Missing'
 'payment_type_annually' 'payment_type_monthly']


## 12.11 Integration of One-Hot Encoded Features

In [98]:
# Create DataFrames for the one-hot encoded features

X_train_encoded_df = pd.DataFrame(
    X_train_encoded,
    columns=one_hot_feature_names,
    index=X_train.index
)

X_test_encoded_df = pd.DataFrame(
    X_test_encoded,
    columns=one_hot_feature_names,
    index=X_test.index
)

# Remove the original categorical columns that were one-hot encoded

X_train_base = X_train.drop(
    columns=one_hot_cols,
    errors="ignore"
)

X_test_base = X_test.drop(
    columns=one_hot_cols,
    errors="ignore"
)

# Add the one-hot encoded features

X_train = pd.concat(
    [X_train_base, X_train_encoded_df],
    axis=1
)

X_test = pd.concat(
    [X_test_base, X_test_encoded_df],
    axis=1
)

print("One-hot encoded features integrated.")

print("\nTraining shape:", X_train.shape)
print("Testing shape:", X_test.shape)

One-hot encoded features integrated.

Training shape: (47520, 334)
Testing shape: (11880, 334)


In [99]:
print("Original one-hot columns remaining in training:")

for col in one_hot_cols:
    print(
        col,
        "->",
        "present" if col in X_train.columns else "removed"
    )

Original one-hot columns remaining in training:
public_meeting -> removed
permit -> removed
quantity -> removed
waterpoint_type_group -> removed
payment_type -> removed
source_type -> removed
waterpoint_type -> removed
extraction_type_class -> removed
water_quality -> removed
basin -> removed
source -> removed
scheme_management -> removed
management -> removed
extraction_type_group -> removed
extraction_type -> removed
region -> removed
region_code -> removed
district_code -> removed
lga -> removed


## 12.12 Final Preprocessing Verification

In [100]:
# Final preprocessing verification

print("Final training shape:", X_train.shape)
print("Final testing shape:", X_test.shape)

print("\nRemaining missing values - training:")
print(X_train.isna().sum().sum())

print("\nRemaining missing values - testing:")
print(X_test.isna().sum().sum())

print("\nNon-numeric columns remaining in training:")
print(X_train.select_dtypes(exclude=["number"]).columns.tolist())

print("\nNon-numeric columns remaining in testing:")
print(X_test.select_dtypes(exclude=["number"]).columns.tolist())

print("Infinite values - training:",
      np.isinf(X_train.select_dtypes(include="number")).sum().sum())

print("Infinite values - testing:",
      np.isinf(X_test.select_dtypes(include="number")).sum().sum())

print("Duplicate columns - training:",
      X_train.columns.duplicated().sum())

print("Duplicate columns - testing:",
      X_test.columns.duplicated().sum())

Final training shape: (47520, 334)
Final testing shape: (11880, 334)

Remaining missing values - training:
0

Remaining missing values - testing:
0

Non-numeric columns remaining in training:
[]

Non-numeric columns remaining in testing:
[]
Infinite values - training: 0
Infinite values - testing: 0
Duplicate columns - training: 0
Duplicate columns - testing: 0


## 12.13 Final Feature Summary

In [101]:
print("Final number of predictors:", X_train.shape[1])
print("Training observations:", X_train.shape[0])
print("Testing observations:", X_test.shape[0])

print("\nFinal feature list:")
print(X_train.columns.tolist())

Final number of predictors: 334
Training observations: 47520
Testing observations: 11880

Final feature list:
['amount_tsh', 'gps_height', 'longitude', 'latitude', 'num_private', 'population', 'recorded_year', 'recorded_month', 'waterpoint_age', 'construction_year_missing', 'funder_freq', 'installer_freq', 'subvillage_freq', 'ward_freq', 'scheme_name_freq', 'public_meeting_False', 'public_meeting_Missing', 'public_meeting_True', 'permit_False', 'permit_Missing', 'permit_True', 'quantity_Missing', 'quantity_dry', 'quantity_enough', 'quantity_insufficient', 'quantity_seasonal', 'waterpoint_type_group_cattle trough', 'waterpoint_type_group_communal standpipe', 'waterpoint_type_group_dam', 'waterpoint_type_group_hand pump', 'waterpoint_type_group_improved spring', 'waterpoint_type_group_other', 'payment_type_Missing', 'payment_type_annually', 'payment_type_monthly', 'payment_type_never pay', 'payment_type_on failure', 'payment_type_other', 'payment_type_per bucket', 'source_type_borehole',

## 12.14 Final Feature-Type Summary

The final predictor dataset contains numerical measurements, numerical code variables, frequency-encoded categorical variables, missingness indicators, and one-hot encoded categorical variables. The target variable is excluded from the predictor matrices. This final structure is used as the input to subsequent model development and evaluation.

In [102]:
print("Final feature count:", X_train.shape[1])

print("\nFeature groups:")

print("\nOriginal numerical measurements:")
print(numeric_measurements)

print("\nNumerical code features:")
print(numeric_code_features)

print("\nFrequency-encoded features:")
print(freq_cols)

print("\nMissingness indicators:")
print([
    col for col in X_train.columns
    if col.endswith("_missing")
])

print("\nOne-hot encoded feature count:")
print(len(one_hot_feature_names))

Final feature count: 334

Feature groups:

Original numerical measurements:
['amount_tsh', 'gps_height', 'longitude', 'latitude', 'num_private', 'population', 'construction_year', 'recorded_year', 'recorded_month', 'waterpoint_age']

Numerical code features:
['region_code', 'district_code']

Frequency-encoded features:
['funder_freq', 'installer_freq', 'subvillage_freq', 'ward_freq', 'scheme_name_freq']

Missingness indicators:
['construction_year_missing']

One-hot encoded feature count:
319


### 12.15 Final Preprocessing Validation

The final predictor datasets were validated after all preprocessing and feature engineering steps. The training and testing datasets contain the same predictor structure, no remaining missing values, and no non-numeric predictor variables.

The final preprocessing process included numerical missing-value treatment, missingness-indicator creation, categorical missing-value treatment, frequency encoding of high-cardinality categorical variables, and one-hot encoding of the remaining categorical variables.

All encoding transformations were fitted using the training data and then applied to the testing data to avoid information leakage.

In [103]:
print("Infinite values in training:", 
      np.isinf(X_train.select_dtypes(include="number")).sum().sum())

print("Infinite values in testing:", 
      np.isinf(X_test.select_dtypes(include="number")).sum().sum())

Infinite values in training: 0
Infinite values in testing: 0


In [104]:
print("Duplicate columns in training:",
      X_train.columns.duplicated().sum())

print("Duplicate columns in testing:",
      X_test.columns.duplicated().sum())

Duplicate columns in training: 0
Duplicate columns in testing: 0


In [105]:
train_only = set(X_train.columns) - set(X_test.columns)
test_only = set(X_test.columns) - set(X_train.columns)

print("Features only in training:", train_only)
print("Features only in testing:", test_only)

Features only in training: set()
Features only in testing: set()


### 12.16 Summary of Final Predictor Dataset

Following preprocessing and feature engineering, the final training dataset contains 47,520 observations and 289 predictor variables, while the testing dataset contains 11,880 observations and the same 289 predictor variables.

The final predictor set contains numerical measurements, numerical code features, frequency-encoded categorical variables, a missingness indicator, and one-hot encoded categorical variables.

High-cardinality categorical variables were frequency encoded using training data only, while the remaining categorical variables were one-hot encoded using an encoder fitted on the training data.

The original high-cardinality categorical variables and original categorical variables selected for one-hot encoding were removed after their encoded representations were integrated.

Final validation confirmed that there are no missing values, no non-numeric predictor columns, and that the training and testing datasets have identical predictor structures.

In [106]:
# Save final preprocessed predictor datasets

X_train.to_csv("../data/processed/X_train_final.csv", index=False)
X_test.to_csv("../data/processed/X_test_final.csv", index=False)

y_train.to_csv("../data/processed/y_train_final.csv", index=False)
y_test.to_csv("../data/processed/y_test_final.csv", index=False)

print("Final preprocessed datasets saved successfully.")

Final preprocessed datasets saved successfully.


In [107]:
import joblib

joblib.dump(
    one_hot_encoder,
    "../models/one_hot_encoder.pkl"
)

joblib.dump(
    frequency_maps,
    "../models/frequency_maps.pkl"
)

print("Encoding objects saved successfully.")

Encoding objects saved successfully.


### 12.17 Duplicate and Invalid Record Verification

The final predictor datasets were checked for duplicate observations and invalid numerical values before model development.

In [108]:
print("Duplicate training rows:", X_train.duplicated().sum())
print("Duplicate testing rows:", X_test.duplicated().sum())

print("Infinite values in training:",
      np.isinf(X_train.select_dtypes(include="number")).sum().sum())

print("Infinite values in testing:",
      np.isinf(X_test.select_dtypes(include="number")).sum().sum())

Duplicate training rows: 254
Duplicate testing rows: 26
Infinite values in training: 0
Infinite values in testing: 0


### 12.18 Investigation of Duplicate Predictor Records

Duplicate predictor rows were identified in the final datasets. These observations are investigated before removal because identical predictor values do not necessarily indicate duplicate records. Multiple water-point observations may legitimately share the same predictor characteristics.

In [109]:
print("Duplicate training rows:", X_train.duplicated().sum())
print("Duplicate testing rows:", X_test.duplicated().sum())

print("\nDuplicate training rows including first occurrence:")
print(X_train[X_train.duplicated(keep=False)].shape)

print("\nDuplicate testing rows including first occurrence:")
print(X_test[X_test.duplicated(keep=False)].shape)

Duplicate training rows: 254
Duplicate testing rows: 26

Duplicate training rows including first occurrence:
(436, 334)

Duplicate testing rows including first occurrence:
(50, 334)


In [110]:
train_check = X_train.copy()
train_check["_target"] = y_train.values

duplicate_target_check = (
    train_check
    .groupby(list(X_train.columns))["_target"]
    .nunique()
)

print("Duplicate predictor groups with different target values:",
      (duplicate_target_check > 1).sum())

Duplicate predictor groups with different target values: 25


### 12.19 Duplicate Record Treatment Decision

Duplicate predictor rows were identified in the final datasets, with 254 duplicate rows in the training predictors and 26 in the testing predictors.

Further investigation showed that 25 groups of identical training predictor values contain different target values. Therefore, the repeated predictor patterns cannot be assumed to represent erroneous duplicate records.

The duplicate observations were consequently retained rather than automatically removed, since removing them could discard valid observations and target variation. No infinite numerical values were identified.

## 12.20 Model-Specific Numerical Scaling

Numerical scaling is model-dependent and is therefore not applied
uniformly to all predictors.

Logistic Regression can be affected by differences in the scale of
numerical features because it is a coefficient-based model. Therefore,
the continuous numerical features will be standardized using
`StandardScaler`.

Tree-based algorithms such as Decision Tree, Random Forest, and
Gradient Boosting do not require numerical scaling because their
splitting decisions are based on feature thresholds rather than
feature magnitude.

Therefore, scaling is prepared specifically for Logistic Regression,
while the unscaled preprocessed dataset is retained for tree-based
models.

The scaler is fitted using the training data only and the same
training-derived parameters are applied to the testing data. This
prevents information leakage.

In [111]:
from sklearn.preprocessing import StandardScaler
import joblib

# Continuous numerical features to be scaled
# Only include features that actually exist in the final dataset
scaling_features = [
    col for col in [
        "amount_tsh",
        "gps_height",
        "longitude",
        "latitude",
        "num_private",
        "population",
        "recorded_year",
        "recorded_month",
        "waterpoint_age"
    ]
    if col in X_train.columns
]

print("Features selected for scaling:")
print(scaling_features)
print("\nNumber of features to scale:", len(scaling_features))

Features selected for scaling:
['amount_tsh', 'gps_height', 'longitude', 'latitude', 'num_private', 'population', 'recorded_year', 'recorded_month', 'waterpoint_age']

Number of features to scale: 9


In [112]:
# Create copies so that the original preprocessed datasets remain unchanged
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# Create and fit scaler using training data only
scaler = StandardScaler()

X_train_scaled[scaling_features] = scaler.fit_transform(
    X_train[scaling_features]
)

# Apply the same training-derived scaler to testing data
X_test_scaled[scaling_features] = scaler.transform(
    X_test[scaling_features]
)

print("Numerical scaling completed.")

Numerical scaling completed.


In [113]:
print("Training means after scaling:")
print(
    X_train_scaled[scaling_features]
    .mean()
    .round(2)
)

print("\nTraining standard deviations after scaling:")
print(
    X_train_scaled[scaling_features]
    .std()
    .round(2)
)

Training means after scaling:
amount_tsh        0.0
gps_height       -0.0
longitude        -0.0
latitude         -0.0
num_private       0.0
population        0.0
recorded_year    -0.0
recorded_month    0.0
waterpoint_age    0.0
dtype: float64

Training standard deviations after scaling:
amount_tsh        1.0
gps_height        1.0
longitude         1.0
latitude          1.0
num_private       1.0
population        1.0
recorded_year     1.0
recorded_month    1.0
waterpoint_age    1.0
dtype: float64


### Scaling Verification

After standardization, the selected numerical features have approximately
zero mean and unit variance in the training dataset.

The scaler was fitted only on the training data. The testing data was
transformed using the same training-derived parameters, ensuring that
information from the testing dataset was not used during preprocessing.

The one-hot encoded categorical variables were not scaled because they
represent categorical membership rather than continuous measurements.

In [114]:
# Save the fitted scaler for use during model development
joblib.dump(
    scaler,
    "../models/standard_scaler.pkl"
)

print("Standard scaler saved successfully.")

Standard scaler saved successfully.


## 12.21 Final Feature Selection Review

Feature selection was performed throughout the preprocessing process
based on the meaning, quality, redundancy, cardinality, and usefulness
of the available predictors.

The following types of features were removed or transformed:

- `id` was removed because it is an observation identifier rather than
  a meaningful predictor.
- `recorded_by` was removed because it contained only one unique value
  and therefore provided no discriminative information.
- `wpt_name` was removed because it behaves like an identifier and has
  very high cardinality.
- Redundant categorical variables were removed after investigating
  relationships between detailed and grouped representations.
- Original high-cardinality categorical variables were removed after
  their information was represented using frequency encoding.
- `date_recorded` was removed after extracting useful temporal features.
- `construction_year` was removed after creating `waterpoint_age` and
  a construction-year missingness indicator.
- `region_code` and `district_code` were treated as categorical
  variables rather than continuous numerical measurements.

These decisions reduce unnecessary or redundant information while
retaining useful predictive information for subsequent model
development.

In [115]:
print("===== Final Feature Selection Review =====\n")

print("1. Identifier removed:")
print("   - id")

print("\n2. Constant feature removed:")
print("   - recorded_by")

print("\n3. Identifier-like feature removed:")
print("   - wpt_name")

print("\n4. Redundant categorical features removed:")
print("   -", redundant_features)

print("\n5. Original high-cardinality features removed:")
print("   -", frequency_encode_cols)

print("\n6. Original date feature removed:")
print("   - date_recorded")

print("\n7. Original construction-year feature removed:")
print("   - construction_year")

print("\nFinal number of predictors:", X_train.shape[1])

===== Final Feature Selection Review =====

1. Identifier removed:
   - id

2. Constant feature removed:
   - recorded_by

3. Identifier-like feature removed:
   - wpt_name

4. Redundant categorical features removed:
   - ['payment', 'quantity_group', 'quality_group', 'management_group', 'source_class']

5. Original high-cardinality features removed:
   - ['funder', 'installer', 'subvillage', 'ward', 'scheme_name']

6. Original date feature removed:
   - date_recorded

7. Original construction-year feature removed:
   - construction_year

Final number of predictors: 334


## 12.22 Final Feature Structure Verification

The final predictor datasets are checked to ensure that training and
testing data contain the same features and that the preprocessing process
has not introduced missing, infinite, non-numeric, or duplicated
predictors.

The final datasets are now ready for model development.

In [116]:
print("===== Final Dataset Verification =====\n")

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

print("\nMissing values in training:")
print(X_train.isna().sum().sum())

print("\nMissing values in testing:")
print(X_test.isna().sum().sum())

print("\nNon-numeric columns in training:")
print(X_train.select_dtypes(exclude=["number"]).columns.tolist())

print("\nNon-numeric columns in testing:")
print(X_test.select_dtypes(exclude=["number"]).columns.tolist())

print("\nInfinite values in training:")
print(
    np.isinf(
        X_train.select_dtypes(include=["number"])
    ).sum().sum()
)

print("\nInfinite values in testing:")
print(
    np.isinf(
        X_test.select_dtypes(include=["number"])
    ).sum().sum()
)

print("\nFeatures only in training:")
print(set(X_train.columns) - set(X_test.columns))

print("\nFeatures only in testing:")
print(set(X_test.columns) - set(X_train.columns))

===== Final Dataset Verification =====

Training shape: (47520, 334)
Testing shape: (11880, 334)

Missing values in training:
0

Missing values in testing:
0

Non-numeric columns in training:
[]

Non-numeric columns in testing:
[]

Infinite values in training:
0

Infinite values in testing:
0

Features only in training:
set()

Features only in testing:
set()


## 12.23 Save Model-Specific Preprocessed Datasets

The unscaled preprocessed datasets are retained for tree-based models,
while the scaled datasets are saved separately for Logistic Regression.

This allows each algorithm to receive the preprocessing appropriate to
its characteristics without unnecessarily scaling models that do not
require it.

In [117]:
# Save unscaled datasets for tree-based models
X_train.to_csv(
    "../data/processed/X_train_final.csv",
    index=False
)

X_test.to_csv(
    "../data/processed/X_test_final.csv",
    index=False
)

# Save scaled datasets for Logistic Regression
X_train_scaled.to_csv(
    "../data/processed/X_train_scaled.csv",
    index=False
)

X_test_scaled.to_csv(
    "../data/processed/X_test_scaled.csv",
    index=False
)

# Save target datasets
y_train.to_csv(
    "../data/processed/y_train_final.csv",
    index=False
)

y_test.to_csv(
    "../data/processed/y_test_final.csv",
    index=False
)

print("All final datasets saved successfully.")

All final datasets saved successfully.
